In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od

In [ ]:
od.download("https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: rija1214
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000


100%|██████████| 5.20G/5.20G [00:51<00:00, 109MB/s] 


In [ ]:
import os, pandas as pd, shutil
DATA_DIR = '/content/skin-cancer-mnist-ham10000'
meta_paths = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if 'metadata' in f.lower() or 'HAM10000' in f]
meta_paths[:3]

['/content/skin-cancer-mnist-ham10000/HAM10000_images_part_2',
 '/content/skin-cancer-mnist-ham10000/HAM10000_metadata.csv',
 '/content/skin-cancer-mnist-ham10000/HAM10000_images_part_1']

In [ ]:
# Load metadata (adapt filename if needed)
meta_file = os.path.join(DATA_DIR, 'HAM10000_metadata.csv')  # update path if different
df = pd.read_csv(meta_file)

# Example HAM10000 dx values: 'nv','mel','bcc','akiec','bkl','df','vasc'
# Map to 4 classes: Melanoma, BCC, SCC (approx from 'akiec'), Benign (everything else)
label_map = {
    'mel': 'melanoma',
    'bcc': 'bcc',
    'akiec': 'scc',   # approximation: actinic keratoses / intraepithelial carcinoma -> map to SCC
    'nv': 'benign',
    'bkl': 'benign',
    'df': 'benign',
    'vasc': 'benign'
}
df['label'] = df['dx'].map(label_map)
df['label'].value_counts()


,count
label,
benign,8061
melanoma,1113
bcc,514
scc,327


In [ ]:
from sklearn.model_selection import train_test_split
import os

# Where images actually live (update)
# Based on the output of the previous cell, the images are in subdirectories within the DATA_DIR.
IMAGES_DIR = DATA_DIR  # change if necessary

# stratified split
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

# create folder tree: dataset/{train,val,test}/{class}/
out_root = '/content/dataset'
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    for cls in split_df['label'].unique():
        os.makedirs(os.path.join(out_root, split_name, cls), exist_ok=True)

# copy images (assumes image files named as in metadata image_id + .jpg)
def copy_split(df_split, split_name):
    for _, r in df_split.iterrows():
        imgid = r['image_id'] if 'image_id' in r else r['image_id'].astype(str)
        # Check in both part_1 and part_2 subdirectories
        for part_dir in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
            for ext in ['.jpg','.jpeg','.png']:
                src = os.path.join(IMAGES_DIR, part_dir, imgid + ext)
                if os.path.exists(src):
                    dst = os.path.join(out_root, split_name, r['label'], os.path.basename(src))
                    if not os.path.exists(dst):
                        shutil.copy(src, dst)
                    break
            if os.path.exists(src):
                break # Found image in one of the parts

copy_split(train_df, 'train')
copy_split(val_df, 'val')
copy_split(test_df, 'test')

print("Done preparing dataset folders under /content/dataset")

Done preparing dataset folders under /content/dataset


In [ ]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 741.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 142.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(out_root,'train'),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical',
    shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(out_root,'val'),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical',
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)


Found 8012 files belonging to 4 classes.
Found 1001 files belonging to 4 classes.
Classes: ['bcc', 'benign', 'melanoma', 'scc']


In [ ]:
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.06),
    tf.keras.layers.RandomZoom(0.08),
])

def prepare(ds, training=False):
    ds = ds.map(lambda x,y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x,y: (data_augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prepare(train_ds, training=True)
val_ds = prepare(val_ds, training=False)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(IMG_SIZE,IMG_SIZE,3), pooling='avg'
)
base.trainable = False

inputs = layers.Input(shape=(IMG_SIZE,IMG_SIZE,3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base(x, training=False)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

ckpt_path = '/content/drive/MyDrive/skin_models/effnetb0_best.h5'  # save to Drive
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

callbacks = [
    ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, verbose=1, restore_best_weights=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)


Epoch 1/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7149 - loss: 0.8882
Epoch 1: val_accuracy improved from -inf to 0.80519, saving model to /content/drive/MyDrive/skin_models/effnetb0_best.h5


251/251 ━━━━━━━━━━━━━━━━━━━━ 316s 1s/step - accuracy: 0.7151 - loss: 0.8877 - val_accuracy: 0.8052 - val_loss: 0.6555 - learning_rate: 1.0000e-04
Epoch 2/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8102 - loss: 0.6342
Epoch 2: val_accuracy improved from 0.80519 to 0.80619, saving model to /content/drive/MyDrive/skin_models/effnetb0_best.h5


251/251 ━━━━━━━━━━━━━━━━━━━━ 289s 1s/step - accuracy: 0.8102 - loss: 0.6343 - val_accuracy: 0.8062 - val_loss: 0.6090 - learning_rate: 1.0000e-04
Epoch 3/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8085 - loss: 0.5903
Epoch 3: val_accuracy improved from 0.80619 to 0.80819, saving model to /content/drive/MyDrive/skin_models/effnetb0_best.h5


251/251 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.8085 - loss: 0.5903 - val_accuracy: 0.8082 - val_loss: 0.5810 - learning_rate: 1.0000e-04
Epoch 4/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8087 - loss: 0.5669
Epoch 4: val_accuracy improved from 0.80819 to 0.80919, saving model to /content/drive/MyDrive/skin_models/effnetb0_best.h5


251/251 ━━━━━━━━━━━━━━━━━━━━ 283s 1s/step - accuracy: 0.8086 - loss: 0.5669 - val_accuracy: 0.8092 - val_loss: 0.5618 - learning_rate: 1.0000e-04
Epoch 5/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 998ms/step - accuracy: 0.8127 - loss: 0.5486
Epoch 5: val_accuracy did not improve from 0.80919
251/251 ━━━━━━━━━━━━━━━━━━━━ 282s 1s/step - accuracy: 0.8126 - loss: 0.5486 - val_accuracy: 0.8092 - val_loss: 0.5485 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 5.


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# prepare test_ds without augmentation
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(out_root,'test'),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode='categorical',
    shuffle=False
)
test_ds = test_ds.map(lambda x,y: (tf.cast(x, tf.float32), y)).prefetch(AUTOTUNE)

y_true = []
y_pred = []
for x,y in test_ds:
    preds = model.predict(x)
    y_pred.extend(np.argmax(preds, axis=1).tolist())
    y_true.extend(np.argmax(y.numpy(), axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=class_names))


Found 1002 files belonging to 4 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 984ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 990ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 996ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 991ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 980ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1000ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 977ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 981ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step    
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 979ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 989ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 980ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 996ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 989ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 989ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 979ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
base_model = model.get_layer('efficientnetb0')

for layer in base_model.layers:
    try:
        if len(layer.output.shape) == 4:
            print(layer.name, layer.output.shape)
    except:
        pass


input_layer_1 (None, 224, 224, 3)
rescaling (None, 224, 224, 3)
normalization (None, 224, 224, 3)
rescaling_1 (None, 224, 224, 3)
stem_conv_pad (None, 225, 225, 3)
stem_conv (None, 112, 112, 32)
stem_bn (None, 112, 112, 32)
stem_activation (None, 112, 112, 32)
block1a_dwconv (None, 112, 112, 32)
block1a_bn (None, 112, 112, 32)
block1a_activation (None, 112, 112, 32)
block1a_se_reshape (None, 1, 1, 32)
block1a_se_reduce (None, 1, 1, 8)
block1a_se_expand (None, 1, 1, 32)
block1a_se_excite (None, 112, 112, 32)
block1a_project_conv (None, 112, 112, 16)
block1a_project_bn (None, 112, 112, 16)
block2a_expand_conv (None, 112, 112, 96)
block2a_expand_bn (None, 112, 112, 96)
block2a_expand_activation (None, 112, 112, 96)
block2a_dwconv_pad (None, 113, 113, 96)
block2a_dwconv (None, 56, 56, 96)
block2a_bn (None, 56, 56, 96)
block2a_activation (None, 56, 56, 96)
block2a_se_reshape (None, 1, 1, 96)
block2a_se_reduce (None, 1, 1, 4)
block2a_se_expand (None, 1, 1, 96)
block2a_se_excite (None, 56, 56

In [ ]:
last_conv_layer_name = "efficientnetb0/top_conv"  # use your last conv layer name here
print("✅ Using:", last_conv_layer_name)


✅ Using: efficientnetb0/top_conv


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import os

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    # Access the EfficientNetB0 base model inside your main model
    base_model = model.get_layer("efficientnetb0")
    last_conv_layer = base_model.get_layer(last_conv_layer_name)

    # Create a model that maps the image input to the activations of the last conv layer
    # and the model's output predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], # Use model.inputs from the main model
        [last_conv_layer.output, model.output] # Use model.output from the main model
    )

    with tf.GradientTape() as tape:
        # Use the main model's input array
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# ✅ Find a suitable convolutional layer inside EfficientNetB0
base_model = model.get_layer("efficientnetb0")
last_conv_layer_name = None
for layer in reversed(base_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

if last_conv_layer_name:
    print("✅ Last conv layer found:", last_conv_layer_name)

    # ✅ Example image for visualization
    sample_img_path = f"/content/dataset/test/{class_names[0]}/{os.listdir(os.path.join('/content/dataset/test', class_names[0]))[0]}"
    img = tf.keras.preprocessing.image.load_img(sample_img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = tf.keras.preprocessing.image.img_to_array(img)
    input_arr = np.expand_dims(img_arr, axis=0)
    input_arr = tf.keras.applications.efficientnet.preprocess_input(input_arr)

    # ✅ Generate Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(input_arr, model, last_conv_layer_name)

    # ✅ Overlay heatmap on original image
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    plt.figure(figsize=(6, 6))
    plt.imshow(img_arr.astype('uint8') / 255.0)
    plt.imshow(heatmap_resized, cmap='jet', alpha=0.4)
    plt.axis('off')
    plt.title("Grad-CAM Visualization")
    plt.show()
else:
    print("Could not find a suitable convolutional layer.")

✅ Last conv layer found: top_conv


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['keras_tensor_243']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


KeyError: "Exception encountered when calling Functional.call().\n\n\x1b[1m138075824392240\x1b[0m\n\nArguments received by Functional.call():\n  • inputs=array([[[[171., 128., 147.],\n         [170., 136., 152.],\n         [177., 144., 161.],\n         ...,\n         [142., 106., 120.],\n         [139., 102., 119.],\n         [140., 101., 119.]],\n\n        [[170., 127., 144.],\n         [171., 128., 148.],\n         [170., 131., 152.],\n         ...,\n         [139., 103., 115.],\n         [140., 104., 118.],\n         [140., 104., 118.]],\n\n        [[170., 127., 144.],\n         [170., 127., 146.],\n         [172., 131., 149.],\n         ...,\n         [140., 102., 115.],\n         [141., 100., 116.],\n         [140., 102., 115.]],\n\n        ...,\n\n        [[159., 117., 127.],\n         [161., 120., 128.],\n         [162., 120., 130.],\n         ...,\n         [167., 127., 136.],\n         [169., 127., 137.],\n         [166., 131., 137.]],\n\n        [[158., 118., 126.],\n        

In [ ]:
import cv2
import glob
import os

# Paths - update if necessary
MASKS_DIR = os.path.join(DATA_DIR, 'masks')  # masks folder
IMG_DIR = IMAGES_DIR
yolo_out = '/content/yolo_dataset'
os.makedirs(yolo_out, exist_ok=True)
for split in ['train','val','test']:
    os.makedirs(os.path.join(yolo_out, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(yolo_out, split, 'labels'), exist_ok=True)

# helper: map label -> int class id
class_to_id = {c:i for i,c in enumerate(sorted(class_names))}
print("class_to_id:", class_to_id)

# Choose which splits to use for detection (we'll reuse the dataset splits created earlier)
def make_yolo_annotations(df_split, split_name):
    for _, r in df_split.iterrows():
        imgid = r['image_id']
        # find image file
        src_img = None
        for ext in ['.jpg','.jpeg','.png']:
            cand = os.path.join(IMAGES_DIR, imgid+ext)
            if os.path.exists(cand):
                src_img = cand
                break
        if src_img is None:
            continue
        # corresponding mask file (update naming if different)
        mask_file = os.path.join(MASKS_DIR, imgid + '_segmentation.png')
        if not os.path.exists(mask_file):
            mask_file = os.path.join(MASKS_DIR, imgid + '.png')  # alternative
        if not os.path.exists(mask_file):
            # no mask -> skip. You could also create a small box around the lesion centroid
            continue

        img = cv2.imread(src_img)
        h, w = img.shape[:2]
        mask = cv2.imread(mask_file, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        # threshold to binary
        _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        yolo_lines = []
        for cnt in contours:
            x,y,bbw,bbh = cv2.boundingRect(cnt)
            # normalize
            x_c = (x + bbw/2)/w
            y_c = (y + bbh/2)/h
            nw = bbw / w
            nh = bbh / h
            class_id = class_to_id[r['label']]
            yolo_lines.append(f"{class_id} {x_c:.6f} {y_c:.6f} {nw:.6f} {nh:.6f}")

        # write label file
        img_dst = os.path.join(yolo_out, split_name, 'images', os.path.basename(src_img))
        lbl_dst = os.path.join(yolo_out, split_name, 'labels', os.path.splitext(os.path.basename(src_img))[0] + '.txt')
        shutil.copy(src_img, img_dst)
        with open(lbl_dst, 'w') as f:
            f.write("\n".join(yolo_lines))

# generate annotations for train/val/test using earlier dataframes
make_yolo_annotations(train_df, 'train')
make_yolo_annotations(val_df, 'val')
make_yolo_annotations(test_df, 'test')

print("YOLO dataset prepared under", yolo_out)


class_to_id: {'bcc': 0, 'benign': 1, 'melanoma': 2, 'scc': 3}
YOLO dataset prepared under /content/yolo_dataset


In [ ]:
# Clone YOLOv5 repo and install requirements
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -q -r requirements.txt
%cd ..

# Create a dataset YAML for YOLOv5
import yaml, os
yolo_yaml = {
    'train': os.path.join('/content/yolo_dataset/train','images'),
    'val': os.path.join('/content/yolo_dataset/val','images'),
    'test': os.path.join('/content/yolo_dataset/test','images'),
    'nc': len(class_names),
    'names': class_names
}
with open('/content/skin_dataset_yolo.yaml', 'w') as f:
    yaml.dump(yolo_yaml, f)

# Start training (this will use GPU)
# Adjust --img, --batch, --epochs for your runtime and dataset size
!python yolov5/train.py --img 640 --batch 16 --epochs 50 --data /content/skin_dataset_yolo.yaml --weights yolov5s.pt --project /content/drive/MyDrive/skin_yolov5 --name yolov5_skin


Cloning into 'yolov5'...
remote: Enumerating objects: 17582, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 17582 (delta 0), reused 0 (delta 0), pack-reused 17579 (from 2)
Receiving objects: 100% (17582/17582), 16.81 MiB | 44.94 MiB/s, done.
Resolving deltas: 100% (11970/11970), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.7/772.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 38.1 MB/s eta 0:00:00
/content
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update

In [ ]:
# Simple IoU function for two boxes in [x1,y1,x2,y2]
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interArea = max(0, xB-xA) * max(0, yB-yA)
    boxAArea = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    boxBArea = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    if interArea == 0: return 0.0
    return interArea / float(boxAArea + boxBArea - interArea)


In [ ]:
# Install gradio if needed
!pip install -q gradio

# load classification model
from tensorflow.keras.models import load_model
clf = load_model('/content/drive/MyDrive/skin_models/effnetb0_best.h5')  # path from earlier

# load YOLOv5 model (use torch hub)
import torch
yolo = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/yolov5s.pt', force_reload=True)

import gradio as gr
import numpy as np
import cv2
import tensorflow as tf

def preprocess_for_clf(img):
    img_res = cv2.resize(img, (IMG_SIZE,IMG_SIZE))
    arr = tf.keras.applications.efficientnet.preprocess_input(np.expand_dims(img_res.astype('float32'),0))
    return arr

def run_all(img):
    np_img = np.array(img)[:,:, :3]
    # classification
    inp = preprocess_for_clf(np_img)
    preds = clf.predict(inp)[0]
    top_idx = int(np.argmax(preds))
    class_name = class_names[top_idx]
    prob = float(preds[top_idx])

    # gradcam
    heatmap = make_gradcam_heatmap(inp, clf, last_conv_layer_name)
    heatmap_on_image = cv2.resize(heatmap, (np_img.shape[1], np_img.shape[0]))
    overlay = (np_img/255.0).copy()
    cmap = plt.get_cmap('jet')
    heatmap_colored = cmap(heatmap_on_image)[:,:,:3]
    overlay = (overlay*0.6 + heatmap_colored*0.4)
    overlay = np.clip(overlay, 0,1)

    # detection
    results = yolo(np_img)
    # results.render()  # will draw boxes in-place (PIL)
    det_img = results.render()[0]  # numpy array with boxes

    return { "Classification": f"{class_name} ({prob:.3f})",
             "GradCAM": overlay,
             "Detection": det_img }

# Build Gradio interface
iface = gr.Interface(
    fn=run_all,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Textbox(label="Classification"),
             gr.Image(type="numpy", label="Grad-CAM"),
             gr.Image(type="numpy", label="Detection")]
)
iface.launch(share=False)


Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip


YOLOv5 🚀 2025-10-7 Python-3.12.11 torch-2.8.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>